In [27]:
import sys
import json
import joblib
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, f1_score

IN_COLAB = "google.colab" in sys.modules

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DATA_DIR = Path("data")
CHECKPOINT_DIR = Path("model_checkpoint")
DATA_DIR.mkdir(exist_ok=True)
CHECKPOINT_DIR.mkdir(exist_ok=True)

DEVICE = "cpu"  # assignment requires this to run CPU-only on a laptop


In [28]:
train_path = DATA_DIR / "train.csv"
public_test_path = DATA_DIR / "public_test.csv"

if IN_COLAB and not (train_path.exists() and public_test_path.exists()):
    from google.colab import files
    print("Upload train.csv and public_test.csv:")
    uploaded = files.upload()
    for fname in uploaded:
        Path(fname).replace(DATA_DIR / fname)

train_df = pd.read_csv(train_path)
public_test_df = pd.read_csv(public_test_path)

print("train shape:", train_df.shape)
print("public_test shape:", public_test_df.shape)
train_df.head()


train shape: (240, 5)
public_test shape: (400, 5)


,id,text,label,label_name,source_file
0,pos_cv230_7428,"well , i'll admit when i first heard about thi...",1,positive,pos/cv230_7428.txt
1,pos_cv853_29233,my summer was recently saved by two very diffe...,1,positive,pos/cv853_29233.txt
2,pos_cv771_28665,in october of 1962 the united states found its...,1,positive,pos/cv771_28665.txt
3,pos_cv449_8785,this is a good year if you want plenty of sci-...,1,positive,pos/cv449_8785.txt
4,pos_cv130_17083,"while watching wes anderson's rushmore , it ma...",1,positive,pos/cv130_17083.txt


In [29]:
print("Train label counts:")
print(train_df["label"].value_counts())
print()
print("Public test label counts:")
print(public_test_df["label"].value_counts())


Train label counts:
label
1    180
0     60
Name: count, dtype: int64

Public test label counts:
label
1    200
0    200
Name: count, dtype: int64


In [30]:
tfidf_pipeline_vectorizer = TfidfVectorizer(
    max_features=6000, ngram_range=(1, 2), min_df=2, sublinear_tf=True
)
X_train_tfidf = tfidf_pipeline_vectorizer.fit_transform(train_df["text"])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
param_grid = {"C": [0.0005, 0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1.0]}

grid = GridSearchCV(
    LogisticRegression(max_iter=3000, class_weight="balanced", random_state=SEED),
    param_grid,
    cv=cv,
    scoring="f1_macro",
)
grid.fit(X_train_tfidf, train_df["label"])

cv_results = pd.DataFrame(grid.cv_results_)[["param_C", "mean_test_score", "std_test_score"]]
cv_results = cv_results.sort_values("mean_test_score", ascending=False)
print(cv_results.to_string(index=False))
print()
print("Selected C (best cross-validated macro-F1):", grid.best_params_["C"])

model_a = grid.best_estimator_


 param_C  mean_test_score  std_test_score
  0.0005         0.685135        0.035784
  0.0010         0.685135        0.035784
  0.0030         0.615807        0.107853
  1.0000         0.597090        0.102381
  0.0100         0.585133        0.094324
  0.0300         0.585133        0.094324
  0.1000         0.585133        0.094324
  0.3000         0.585133        0.094324

Selected C (best cross-validated macro-F1): 0.0005


In [31]:
train_split_df, val_split_df = train_test_split(
    train_df, test_size=0.15, stratify=train_df["label"], random_state=SEED
)

vectorizer_b = TfidfVectorizer(max_features=6000, ngram_range=(1, 2), min_df=2, sublinear_tf=True)
X_train_split_tfidf = vectorizer_b.fit_transform(train_split_df["text"])
X_val_split_tfidf = vectorizer_b.transform(val_split_df["text"])
X_public_test_tfidf_b = vectorizer_b.transform(public_test_df["text"])

svd = TruncatedSVD(n_components=100, random_state=SEED)
X_train_svd = svd.fit_transform(X_train_split_tfidf)
X_val_svd = svd.transform(X_val_split_tfidf)
X_public_test_svd = svd.transform(X_public_test_tfidf_b)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_svd)
X_val_scaled = scaler.transform(X_val_svd)
X_public_test_scaled = scaler.transform(X_public_test_svd)


def to_tensor(x, dtype=torch.float32):
    return torch.tensor(x, dtype=dtype)


X_train_t = to_tensor(X_train_scaled)
y_train_t = to_tensor(train_split_df["label"].to_numpy())
X_val_t = to_tensor(X_val_scaled)
y_val_t = to_tensor(val_split_df["label"].to_numpy())


class SentimentMLP(nn.Module):
    def __init__(self, input_dim=100, hidden_dim=32, dropout=0.4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


n_pos = (train_split_df["label"] == 1).sum()
n_neg = (train_split_df["label"] == 0).sum()
pos_weight = torch.tensor([n_neg / n_pos], dtype=torch.float32)

model_b = SentimentMLP(input_dim=X_train_scaled.shape[1]).to(DEVICE)
optimizer = torch.optim.Adam(model_b.parameters(), lr=1e-3, weight_decay=1e-4)
loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

BATCH_SIZE = 16
MAX_EPOCHS = 300
PATIENCE = 15

best_val_loss = float("inf")
best_state = None
epochs_without_improvement = 0
history = {"train_loss": [], "val_loss": []}

n = X_train_t.shape[0]
for epoch in range(MAX_EPOCHS):
    model_b.train()
    perm = torch.randperm(n)
    epoch_loss = 0.0
    for i in range(0, n, BATCH_SIZE):
        idx = perm[i : i + BATCH_SIZE]
        xb, yb = X_train_t[idx], y_train_t[idx]
        optimizer.zero_grad()
        logits = model_b(xb)
        loss = loss_fn(logits, yb)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * len(idx)
    train_loss = epoch_loss / n

    model_b.eval()
    with torch.no_grad():
        val_loss = loss_fn(model_b(X_val_t), y_val_t).item()

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = {k: v.clone() for k, v in model_b.state_dict().items()}
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= PATIENCE:
        print(f"Early stopping at epoch {epoch} (best val_loss={best_val_loss:.4f})")
        break

model_b.load_state_dict(best_state)

print(f"Best validation loss: {best_val_loss:.4f}")
print(f"Final train_loss: {history['train_loss'][-1]:.4f}")
print(f"Final val_loss:   {history['val_loss'][-1]:.4f}")


Early stopping at epoch 43 (best val_loss=0.2639)
Best validation loss: 0.2639
Final train_loss: 0.0217
Final val_loss:   0.2788


In [32]:
model_b.eval()
with torch.no_grad():
    test_logits_b = model_b(to_tensor(X_public_test_scaled))
    test_probs_b = torch.sigmoid(test_logits_b).numpy()
test_preds_b = (test_probs_b >= 0.5).astype(int)

acc_b = accuracy_score(public_test_df["label"], test_preds_b)
cm_b = confusion_matrix(public_test_df["label"], test_preds_b)
print(f"Model B public test accuracy: {acc_b:.4f}")
print(classification_report(public_test_df["label"], test_preds_b, target_names=["negative", "positive"]))
print("Confusion matrix:\n", cm_b)


Model B public test accuracy: 0.7025
              precision    recall  f1-score   support

    negative       0.89      0.47      0.61       200
    positive       0.64      0.94      0.76       200

    accuracy                           0.70       400
   macro avg       0.76      0.70      0.68       400
weighted avg       0.76      0.70      0.68       400

Confusion matrix:
 [[ 93 107]
 [ 12 188]]


In [33]:
test_preds_a = model_a.predict(tfidf_pipeline_vectorizer.transform(public_test_df["text"]))

acc_a = accuracy_score(public_test_df["label"], test_preds_a)
cm_a = confusion_matrix(public_test_df["label"], test_preds_a)
print(f"Model A public test accuracy: {acc_a:.4f}")
print(classification_report(public_test_df["label"], test_preds_a, target_names=["negative", "positive"]))

cm_a_df = pd.DataFrame(cm_a, index=["true negative", "true positive"], columns=["pred negative", "pred positive"])
cm_b_df = pd.DataFrame(cm_b, index=["true negative", "true positive"], columns=["pred negative", "pred positive"])
print("Model A (TF-IDF + Logistic Regression) confusion matrix:")
print(cm_a_df)
print()
print("Model B (TF-IDF + SVD + Neural network) confusion matrix:")
print(cm_b_df)
print()
print(f"Model A (TF-IDF + Logistic Regression): accuracy = {acc_a:.4f}")
print(f"Model B (TF-IDF + SVD + Neural network): accuracy = {acc_b:.4f}")


Model A public test accuracy: 0.7150
              precision    recall  f1-score   support

    negative       0.84      0.53      0.65       200
    positive       0.66      0.90      0.76       200

    accuracy                           0.71       400
   macro avg       0.75      0.72      0.70       400
weighted avg       0.75      0.71      0.70       400

Model A (TF-IDF + Logistic Regression) confusion matrix:
               pred negative  pred positive
true negative            106             94
true positive             20            180

Model B (TF-IDF + SVD + Neural network) confusion matrix:
               pred negative  pred positive
true negative             93            107
true positive             12            188

Model A (TF-IDF + Logistic Regression): accuracy = 0.7150
Model B (TF-IDF + SVD + Neural network): accuracy = 0.7025


In [34]:
if acc_a >= acc_b:
    winner = "logistic_regression"
    print(f"Selected Model A (Logistic Regression), accuracy={acc_a:.4f} >= Model B accuracy={acc_b:.4f}")

    joblib.dump(tfidf_pipeline_vectorizer, CHECKPOINT_DIR / "tfidf_vectorizer.joblib")
    joblib.dump(model_a, CHECKPOINT_DIR / "logreg_classifier.joblib")
    final_preds = test_preds_a
    final_acc = acc_a
else:
    winner = "neural_network"
    print(f"Selected Model B (Neural network), accuracy={acc_b:.4f} > Model A accuracy={acc_a:.4f}")

    joblib.dump(vectorizer_b, CHECKPOINT_DIR / "tfidf_vectorizer.joblib")
    joblib.dump(svd, CHECKPOINT_DIR / "svd.joblib")
    joblib.dump(scaler, CHECKPOINT_DIR / "scaler.joblib")
    torch.save(model_b.state_dict(), CHECKPOINT_DIR / "mlp_classifier.pt")
    final_preds = test_preds_b
    final_acc = acc_b

config = {
    "winning_model": winner,
    "public_test_accuracy": float(final_acc),
    "seed": SEED,
    "decision_threshold": 0.5,
}
with open(CHECKPOINT_DIR / "config.json", "w") as f:
    json.dump(config, f, indent=2)

print("Saved checkpoint files:", [p.name for p in CHECKPOINT_DIR.iterdir()])


Selected Model A (Logistic Regression), accuracy=0.7150 >= Model B accuracy=0.7025
Saved checkpoint files: ['tfidf_vectorizer.joblib', 'config.json', 'logreg_classifier.joblib']


In [35]:
predictions_df = pd.DataFrame({
    "id": public_test_df["id"],
    "predicted_label": final_preds,
})
predictions_df.to_csv("public_test_predictions.csv", index=False)
predictions_df.head()


,id,predicted_label
0,pos_cv696_29740,1
1,pos_cv669_22995,1
2,neg_cv963_7208,1
3,pos_cv182_7281,1
4,pos_cv162_10424,0


In [36]:
with open(CHECKPOINT_DIR / "config.json") as f:
    loaded_config = json.load(f)

reloaded_vectorizer = joblib.load(CHECKPOINT_DIR / "tfidf_vectorizer.joblib")

if loaded_config["winning_model"] == "logistic_regression":
    reloaded_clf = joblib.load(CHECKPOINT_DIR / "logreg_classifier.joblib")
    check_preds = reloaded_clf.predict(reloaded_vectorizer.transform(public_test_df["text"][:5]))
else:
    reloaded_svd = joblib.load(CHECKPOINT_DIR / "svd.joblib")
    reloaded_scaler = joblib.load(CHECKPOINT_DIR / "scaler.joblib")
    reloaded_model = SentimentMLP(input_dim=reloaded_svd.n_components)
    reloaded_model.load_state_dict(torch.load(CHECKPOINT_DIR / "mlp_classifier.pt"))
    reloaded_model.eval()
    feats = reloaded_scaler.transform(reloaded_svd.transform(reloaded_vectorizer.transform(public_test_df["text"][:5])))
    with torch.no_grad():
        check_preds = (torch.sigmoid(reloaded_model(to_tensor(feats))) >= 0.5).int().numpy()

print("Reloaded-model predictions on first 5 public test examples:", [int(x) for x in check_preds])
print("Original predictions on the same examples:               ", [int(x) for x in final_preds[:5]])
assert list(check_preds) == list(final_preds[:5]), "Reloaded model predictions do not match!"
print("Reload check passed.")


Reloaded-model predictions on first 5 public test examples: [1, 1, 1, 1, 0]
Original predictions on the same examples:                [1, 1, 1, 1, 0]
Reload check passed.


In [37]:
if IN_COLAB:
    import shutil
    from google.colab import files

    shutil.make_archive("model_checkpoint", "zip", CHECKPOINT_DIR)
    files.download("model_checkpoint.zip")
    files.download("public_test_predictions.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>